In [0]:
from pyspark.sql.functions import col, trim, lower, initcap, when, count

# Analyze cust_info table

In [0]:
df = spark.read.table("workspace.bronze.crm_cust_info")

In [0]:
df = df.withColumnsRenamed(
    {
        'cst_id': 'id',
        'cst_key': 'customer_key',
        'cst_firstname': 'firstname',
        'cst_lastname': 'lastname',
        'cst_marital_status': 'marital_status',
        'cst_gndr': 'gender',
        'cst_create_date': 'create_date'
    }
)

In [0]:
df.describe()

In [0]:
df.select(
    count(when(col("id").isNull(), col("id"))).alias("null_id_count"),
    count(when(col("customer_key").isNull(), col("customer_key"))).alias("null_customer_key_count")
).display()

In [0]:
df.display()

In [0]:
# drop duplicates
df = df.distinct()

In [0]:
def clean_string_column(column_name):
    c = col(column_name)
    c = trim(c)         
    c = lower(c)
    c = initcap(c)        
    return c.alias(column_name)

string_trim_cols = {
    col_name: clean_string_column(col_name)
    for col_name, dtype in df.dtypes if dtype == 'string'
}

df = df.withColumns(string_trim_cols) 
display(df.limit(10))

In [0]:
df = df.withColumn(
        'marital_status',
        when(col('marital_status') == 'M', 'Married')
        .when(col('marital_status') == 'S', 'Single')
        .otherwise(col('marital_status'))
)
df.display()

In [0]:
df.select("marital_status").distinct().display()

In [0]:

df = df.withColumn(
    'gender', 
    when(col('gender') == 'M', 'Male')
    .when(col('gender') == 'F', 'Female')
    .otherwise(col('gender'))
)
df.display()


In [0]:
df.select("gender").distinct().display()

In [0]:
df.selectExpr("MAX(create_date) AS max_create_date", "MIN(create_date) as min_create_date").display()

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.crm_customers')

Check if it was saved correctly

In [0]:
df = spark.read.table('workspace.silver.crm_customers')
df.display()